## Divisão da planilha 'dataset_long.csv' em 7 novas planilhas

In [2]:
import pandas as pd

from pathlib import Path

# ---- Configuração ----
INPUT_FILE = Path('../data/dataset_long.csv')
# Os sete CSVs gerados ficam diretamente em data/, na raiz do projeto.
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colunas removidas de todos os arquivos de saída, além de 'key_area'
# (decisão do grupo — ver Card H)
COLUMNS_TO_DROP = ['run_id', 'tc_number', 'reason', 'active_limit', 'key_area']

# Componente excluído do Mainback: identificado como erro de sensor
# (ver discussão de outliers / seção 4.2.2 da documentação)
MAINBACK_EXCLUDE_TC = 'Oven RTD Connector Body'

In [3]:
def split_by_key_area(df: pd.DataFrame, output_dir: Path) -> dict:

    summary = {}
    key_areas = df['key_area'].unique()
    print(f'Áreas encontradas ({len(key_areas)}): {list(key_areas)}\n')

    for area in key_areas:
        filename = output_dir / f"{area.lower().replace(' ', '_')}_data.csv"
        area_df = df[df['key_area'] == area].copy()

        if area == 'Mainback' and 'tc_name' in area_df.columns:
            area_df = area_df[area_df['tc_name'] != MAINBACK_EXCLUDE_TC]

        area_df = area_df.drop(columns=[c for c in COLUMNS_TO_DROP if c in area_df.columns])

        if 'temp' in area_df.columns:
            ordered_cols = [c for c in area_df.columns if c != 'temp'] + ['temp']
            area_df = area_df[ordered_cols]

        area_df.to_csv(filename, index=False)
        summary[area] = len(area_df)
        print(f'"{filename.name}" salvo com {len(area_df)} linhas.')

    return summary

In [4]:
df = pd.read_csv(INPUT_FILE)
print(f"Dataset principal carregado: {len(df)} linhas, {df['key_area'].nunique()} key_areas.\n")

summary = split_by_key_area(df, OUTPUT_DIR)

Dataset principal carregado: 4065 linhas, 7 key_areas.

Áreas encontradas (7): ['Mainback', 'Coil Temps', 'Main Vent Sys', 'IPC Cooling', 'IPC Comp', 'Oven HMI', 'CT Knobs']

"mainback_data.csv" salvo com 557 linhas.
"coil_temps_data.csv" salvo com 420 linhas.
"main_vent_sys_data.csv" salvo com 288 linhas.
"ipc_cooling_data.csv" salvo com 218 linhas.
"ipc_comp_data.csv" salvo com 1979 linhas.
"oven_hmi_data.csv" salvo com 220 linhas.
"ct_knobs_data.csv" salvo com 330 linhas.


In [5]:
# ---- Verificação de integridade ----
total_saved = sum(summary.values())
excluded_rows = len(df[df['tc_name'] == MAINBACK_EXCLUDE_TC]) if 'tc_name' in df.columns else 0
expected = len(df) - excluded_rows

print('Resumo por área:')
for area, n in summary.items():
    print(f'  {area}: {n} linhas')

print(f"\nTotal salvo: {total_saved} | Esperado (total original - excluídas): {expected}")
assert total_saved == expected, 'Divergência na contagem de linhas! Verifique o processamento.'
print('Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.')

Resumo por área:
  Mainback: 557 linhas
  Coil Temps: 420 linhas
  Main Vent Sys: 288 linhas
  IPC Cooling: 218 linhas
  IPC Comp: 1979 linhas
  Oven HMI: 220 linhas
  CT Knobs: 330 linhas

Total salvo: 4012 | Esperado (total original - excluídas): 4012
Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.
